In [ ]:
%load_ext autoreload
%autoreload 2

# Finding Better Prompts from Retrieved Examples

## Goal

This notebook runs the complete BSc experiment:

```text
Bias-in-Bios → hold out one column → retrieve examples → build prompts
→ score every allowed label as a continuation of each prompt
→ select one prompt per language model on validation rows
→ evaluate each selected prompt once on separate test rows
→ inspect predictive quality and group disparities
```

`hard_text` is always input. With target **profession**, gender is the other input and audit group. With target **gender**, profession is the other input and audit group. The structured target value is never included in a validation or test prompt.

Every candidate label is scored against the same master prompt, examples, query, and chat wrapper.

## Setup and assumptions

Use this DataSpell interpreter:

```text
/opt/homebrew/anaconda3/envs/prompt-selection/bin/python
```

Install `requirements.txt` once. Edit `config.yaml` before running if you want a different target, data size, retrieval method, number/order of examples, master prompt, language model, or validation ranking objective.

Language model inference uses Hugging Face Transformers directly. Language models are loaded and released sequentially. Retrieval encoders use SentenceTransformers.

The first real run can download language model files and builds one persistent LanceDB table per embedding model. Matching later runs reuse those tables instead of embedding the training pool again. Delete `data/lancedb/` before changing data or embedding settings. Semantic conditions compare Qwen3-Embedding-8B and BAAI/bge-large-en-v1.5; training documents remain raw `hard_text`, while each encoder receives its own query prefix.

The `label_score` policy computes a mean conditional token log-score for every allowed label and chooses the largest. Scores are relative rankings rather than calibrated probabilities, and inference errors stop the run.

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

raw_train = load_dataset(
    "LabHC/bias_in_bios",
    split="train",
)

print(len(raw_train))  # 257478
print(raw_train[0])

In [ ]:
from modeling import load_language_model, clear_language_model_memory
import inspect

# a = ['Qwen/Qwen3.6-27B', 'Qwen/Qwen3.5-27B', 'Qwen/Qwen3.6-35B-A3B', 'google/gemma-4-31B-it']
# a = ['google/gemma-4-31B-it']
# b = ['6a9e13bd6fc8f0983b9b99948120bc37f49c13e9', 'fc05daec18b0a78c049392ed2e771dde82bdf654',
#      '995ad96eacd98c81ed38be0c5b274b04031597b0', '842da3794eaa0b77d5f08bae87a17459d91ff475']
# b = ['842da3794eaa0b77d5f08bae87a17459d91ff475']
# c = 'bfloat16'
#
# for x, y in zip(a, b):
#     tokenizer, language_model = load_language_model(x, y, 'mps', c)
#     forward_parameters = inspect.signature(language_model.forward).parameters
#     print(language_model.name_or_path, 'logits_to_keep' in forward_parameters)
#
#     del language_model, tokenizer
#     clear_language_model_memory('mps')

In [ ]:
tokenizer, language_model = load_language_model('Qwen/Qwen3.6-27B', '6a9e13bd6fc8f0983b9b99948120bc37f49c13e9', 'mps', c)

In [ ]:
{'hi': 1, 'how': 2}.__getitem__

In [ ]:
from pathlib import Path
import sys

import yaml
from IPython.display import Image, display

# DataSpell can start in this folder or in its parent project folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    candidate = PROJECT_ROOT / "retrieval-guided-master-prompt-selection"
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError("Could not find config.yaml")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dataset import load_data, task_settings  # noqa: E402
from evaluation import (  # noqa: E402
    ACCURACY_METRIC_COLUMN,
    MACRO_RECALL_METRIC_COLUMN,
    resolve_metric_column,
)
from modeling import render_input  # noqa: E402
from pipeline import load_config, run_experiment, validate_config  # noqa: E402

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(CONFIG_PATH)
validate_config(config)
DISPLAY_LIMIT = 40
target, audit_column, professions, target_labels = task_settings(config)

language_model_configs = config["inference"]["language_models"]
language_model_count = len(language_model_configs)
per_language_model_condition_count = (
        len(config["retrieval"]["methods"])
        * len(config["retrieval"]["embedding_models"])
        * len(config["retrieval"]["example_counts"])
        * len(config["retrieval"]["example_orders"])
        * len(config["prompt_templates"])
)
condition_count = language_model_count * per_language_model_condition_count
validation_row_count = (
        len(professions)
        * 2
        * int(config["dataset"]["validation_per_profession_gender"])
)
test_row_count = (
        len(professions)
        * 2
        * int(config["dataset"]["test_per_profession_gender"])
)

print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
print(f"Held-out target: {target}")
print(f"Visible input columns: hard_text + {audit_column}")
print(f"Allowed answers: {target_labels}")
print("Configured language models:")
for language_model_config in language_model_configs:
    print(f"- {language_model_config['id']}")
print(f"Validation conditions per language model: {per_language_model_condition_count}")
print(f"Total validation conditions: {condition_count}")
print(f"Validation rows per condition: {validation_row_count}")
print(f"Final-test rows per selected language model condition: {test_row_count}")
print(
    "Row-condition evaluations:",
    condition_count * validation_row_count + language_model_count * test_row_count,
)
print("Each allowed label is scored directly with the configured Hugging Face language model.")

## Step 1 — Check the three data partitions and visible fields

Training rows form the retrieval pool. Balanced validation cells select the prompt; disjoint balanced test cells estimate its final performance. The next cell loads the data cache—or downloads missing rows—and shows exactly what may enter a query prompt; it does not load a language model or embedding model.

In [ ]:
train_rows, validation_rows, test_rows_data, dataset_counts = load_data(config, PROJECT_ROOT)

dataset_counts

In [ ]:
first_rendered_query = render_input(validation_rows[0], target)
print("First language-model-visible validation query:")
print(first_rendered_query)
print(
    f"True {target} is stored separately for evaluation:",
    validation_rows[0][target],
)

## Step 2 — Select within each language model, then evaluate once on test

Every prompt condition receives the same validation rows. The configured metric ranks conditions separately within each language model. Exactly one winner per language model is then inferred on untouched test rows, avoiding selection on the final evaluation set and keeping every language model's search budget equal.

In [ ]:
run = run_experiment(config, PROJECT_ROOT, progress=print)
best_prompts_path = run["best_prompts"]
print("Results folder:", run["run_dir"])
print("Selected prompts file:", best_prompts_path)

language_model_ids = {entry["id"] for entry in language_model_configs}
selected_validation = run["validation_results"].loc[
    run["validation_results"]["selected_for_test"].astype(bool)
]
assert len(selected_validation) == language_model_count
assert set(selected_validation["language_model"]) == language_model_ids
assert len(run["results"]) == language_model_count
assert set(run["results"]["language_model"]) == language_model_ids
assert set(run["predictions"]["predicted_label"]).issubset(target_labels)
print("Selection check passed: one validation winner and one final-test result per language model.")
print("Closed-set check passed: every prediction is an allowed label.")

## Step 3 — Compare validation conditions and read the final results

`validation_results` ranks conditions within each language model and marks one winner per language model. `results` contains one independent final-test row for every selected language model condition. The plot gallery covers every numeric validation summary and provides final-test summary, class, group, fairness, coverage, and confusion diagnostics.

In [ ]:
ranking_metric = resolve_metric_column(config["defaults"]["ranking_metric"])
summary_columns = [
    "language_model", "rank", "selected_for_test", "condition",
    ACCURACY_METRIC_COLUMN, "macro_f1", MACRO_RECALL_METRIC_COLUMN,
    "matthews_correlation_coefficient", "cohen_kappa",
    "worst_group_accuracy", "group_accuracy_difference",
    "max_demographic_parity_difference",
    "max_equal_opportunity_difference",
    "max_equalized_odds_difference",
]
if ranking_metric not in summary_columns:
    summary_columns.append(ranking_metric)
summary_columns = [
    column for column in summary_columns if column in run["validation_results"].columns
]
print("Validation prompt ranking (rank resets within each language model):")
display(
    run["validation_results"]
    .sort_values(["language_model", "rank"], kind="stable")[summary_columns]
)

final_columns = [
    "language_model", "condition",
    ACCURACY_METRIC_COLUMN, "macro_f1", MACRO_RECALL_METRIC_COLUMN,
    "matthews_correlation_coefficient", "cohen_kappa",
    "worst_group_accuracy", "group_accuracy_difference",
    "max_demographic_parity_difference",
    "max_equal_opportunity_difference",
    "max_equalized_odds_difference",
]
if ranking_metric not in final_columns:
    final_columns.append(ranking_metric)
final_columns = [column for column in final_columns if column in run["results"].columns]
print("Independent final-test result for each language model:")
display(run["results"].sort_values("language_model", kind="stable")[final_columns])

for plot_name, plot_path in run["plots"].items():
    print(plot_name.replace("_", " ").title())
    display(Image(filename=str(plot_path)))

## Step 4 — Inspect final-test denominators and failure modes

The summary is not enough by itself. Per-class scores show which labels fail, and group rates show the supports behind each disparity. The next cell displays final-test details for every selected language model condition; all validation details are in the same returned tables with `evaluation_split == "validation"`.

In [ ]:
test_detail_tables = {
    "Per-class metrics": run["class_metrics"].query("evaluation_split == 'test'"),
    "Group disparities": run["fairness_metrics"].query("evaluation_split == 'test'"),
    "Group rates": run["group_metrics"].query("evaluation_split == 'test'"),
    "Confusion counts": run["confusion_matrix"].query("evaluation_split == 'test'"),
}
for table_name, table in test_detail_tables.items():
    preview = table.groupby("language_model", sort=False, group_keys=False).head(DISPLAY_LIMIT)
    print(
        f"{table_name}: showing {len(preview)} of {len(table)} rows "
        f"(up to {DISPLAY_LIMIT} per language model)"
    )
    display(preview)

## Metric and label-scoring guide

### Notation

$c$ is a target class, $g$ is an audit group, $K$ is the number of target classes, $N$ is the total number of evaluated rows, $N_g$ is the size of group $g$, and $n_c$ is the true support of class $c$. $D_m$ is the set of classes where metric $m$ is defined. $TP$, $FP$, $FN$, and $TN$ are one-vs-rest true-positive, false-positive, false-negative, and true-negative counts.

### Classification metrics

$$Precision_c=PPV_c=\frac{TP_c}{TP_c+FP_c},\quad
Recall_c=TPR_c=\frac{TP_c}{TP_c+FN_c},\quad
F1_c=\frac{2TP_c}{2TP_c+FP_c+FN_c}$$

$$Specificity_c=TNR_c=\frac{TN_c}{TN_c+FP_c},\quad
FPR_c=\frac{FP_c}{FP_c+TN_c},\quad
FNR_c=\frac{FN_c}{FN_c+TP_c},\quad
NPV_c=\frac{TN_c}{TN_c+FN_c}$$

For any class metric $m_c$, undefined values are omitted from its aggregate:

$$Accuracy=\frac{\sum_cTP_c}{N},\quad
Macro(m)=\frac{1}{|D_m|}\sum_{c\in D_m}m_c,\quad
Weighted(m)=\frac{\sum_{c\in D_m}n_cm_c}{\sum_{c\in D_m}n_c}$$

When $m$ is defined for all classes, $|D_m|=K$.

In this single-label multiclass task, let $T=\sum_cTP_c$ be the number of correct rows and $E=\sum_cFP_c=\sum_cFN_c$ the number of errors. Since $N=T+E$:

$$MicroPrecision=\frac{T}{T+E}=MicroRecall=MicroF1=Accuracy$$

$$WeightedRecall=\frac{1}{N}\sum_{c:n_c>0}n_c\frac{TP_c}{n_c}
=\frac{\sum_cTP_c}{N}=Accuracy,\qquad
BalancedAccuracy=\frac{1}{|D_R|}\sum_{c\in D_R}Recall_c=MacroRecall$$

$D_R$ is the set of classes whose recall is defined.

These identities follow from the formulas, not from a particular result. Each equality family therefore has one result column containing all of its standard names.

For the multiclass confusion matrix $C$, let $s=\sum_{ij}C_{ij}$, $q=\operatorname{trace}(C)$, $p_k=\sum_iC_{ik}$ be the predicted total for class $k$, and $t_k=\sum_jC_{kj}$ be its true total:

$$MCC=\frac{qs-\sum_kp_kt_k}
{\sqrt{(s^2-\sum_kp_k^2)(s^2-\sum_kt_k^2)}}$$

For Cohen's kappa, $p_o=Accuracy$ is observed agreement and $p_e=\sum_k(t_k/s)(p_k/s)$ is chance-expected agreement:

$$\kappa=\frac{p_o-p_e}{1-p_e}$$

### Group and fairness metrics

For class $c$ within audit group $g$:

$$SR_{c,g}=\frac{TP_{c,g}+FP_{c,g}}{N_g},\quad
TPR_{c,g}=\frac{TP_{c,g}}{TP_{c,g}+FN_{c,g}},\quad
FPR_{c,g}=\frac{FP_{c,g}}{FP_{c,g}+TN_{c,g}}$$

$$PPV_{c,g}=\frac{TP_{c,g}}{TP_{c,g}+FP_{c,g}},\qquad
Accuracy_g=\frac{\#\text{ correct rows in }g}{N_g}$$

Let $Range_g(x)=\max_gx_g-\min_gx_g$. Then:

$$DPDiff_c=Range_g(SR_{c,g}),\quad
DPRatio_c=\frac{\min_gSR_{c,g}}{\max_gSR_{c,g}},\quad
EqualOpportunityDiff_c=Range_g(TPR_{c,g})$$

$$FPRDiff_c=Range_g(FPR_{c,g}),\quad
EqualizedOddsDiff_c=\max(EqualOpportunityDiff_c,FPRDiff_c),\quad
PredictiveParityDiff_c=Range_g(PPV_{c,g})$$

$WorstGroupAccuracy=\min_gAccuracy_g$ and $GroupAccuracyDiff=Range_g(Accuracy_g)$. For classwise disparity $d_c$, let $D_d$ be the classes where it is defined:

$$MeanDisparity(d)=\frac{1}{|D_d|}\sum_{c\in D_d}d_c,\quad
WorstDifference(d)=\max_{c\in D_d}d_c,\quad
WorstDPRatio=\min_{c\in D_d}DPRatio_c$$

The corresponding defined-class count is $|D_d|$. A zero denominator is stored as blank/`NaN`, and a disparity requires at least two defined groups.

### Label scoring

For allowed label $c$, $T_c$ is its token sequence, $t_j$ is its $j$-th token, and $t_{<j}$ denotes its earlier tokens:

$$score(c)=\frac{1}{|T_c|}\sum_j
\log P(t_j\mid prompt,t_{<j}),\qquad
\hat c=\arg\max_{c\in labels}score(c)$$

Mean normalization reduces the automatic disadvantage of multi-token labels. These are relative ranking scores, not calibrated probabilities. Closed-set choice guarantees a configured label; it does not guarantee correctness or fairness.

## Step 5 — Inspect selected prompts and their test predictions

`best_prompts.txt` contains one resolved validation-selected master instruction per language model, with hyperparameters, validation score, and final-test score. Full prompts vary by query because their retrieved examples differ, so they remain in `predictions.csv`.

`label_scores` is a JSON mapping from each allowed label to its mean conditional token log-score.

In [ ]:
print(Path(best_prompts_path).read_text(encoding="utf-8"))
selected_conditions = set(selected_validation["condition"])
prediction_columns = [
    "evaluation_split", "query_id", "target", "true_label", "audit_group",
    "predicted_label", "language_model", "condition", "retrieval_method",
    "embedding_model", "example_count", "example_order", "prompt_name", "label_scores",
]
prediction_columns = [
    column for column in prediction_columns if column in run["predictions"].columns
]
selected_test_predictions = run["predictions"].loc[
    run["predictions"]["evaluation_split"].eq("test")
    & run["predictions"]["condition"].isin(selected_conditions)
    ]
selected_test_preview = (
    selected_test_predictions
    .groupby("language_model", sort=False, group_keys=False)
    .head(DISPLAY_LIMIT)
)
print(
    f"Selected test predictions: showing {len(selected_test_preview)} of "
    f"{len(selected_test_predictions)} rows "
    f"(up to {DISPLAY_LIMIT} per language model across {language_model_count} language models)"
)
display(selected_test_preview[prediction_columns])

## Next run

Change `defaults.target` from `profession` to `gender` and run all cells again. The same prompt candidates are searched, but each target and language model receives its own validation winner. Do not directly compare the two task scores as though they had identical meanings: they have different class sets, and profession is an audit subgroup—not a protected attribute—when gender is the target.

Use the configured validation and test cell sizes for thesis runs. Closed-set label scoring is deterministic, so changing inference seeds does not create independent language model evidence.

The optional Gradio UI exposes the same YAML, guidance, tables, and plot and calls this exact pipeline. Run `app.py` in DataSpell when needed.